# FISH Coordinate Imputation Tutorial

This tutorial demonstrates how to impute missing 3D coordinates in multiplexed DNA FISH data using uchrom's `im.impute` module.

## Overview

Multiplexed FISH experiments often have ~20-40% missing coordinates due to:
- Weak signals
- Probe overlap
- Optical issues

We provide three imputation methods:
1. **Linear interpolation** - Fast, no extra dependencies
2. **Cubic spline** - Smooth, no extra dependencies  
3. **SnapFISH-IMPUTE** - Similarity-weighted, best accuracy (requires sklearn, skimage)

In [ ]:
import uchrom as uc
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Test Data

We'll use a small subset of mESC Sox2 data (Huang et al. 2021) with 5 cells and ~30% missing coordinates.

In [ ]:
# Load data with missing coordinates
cd = uc.ChromData.read('example-data/mESC_Sox2_5cells_wnan.h5cd')

print(f"Data shape: {cd.coords.shape}")
print(f"Cells: {cd.spots['trace_id'].nunique()}")
print(f"Loci per cell: {len(cd.spots) // cd.spots['trace_id'].nunique()}")

n_missing = np.isnan(cd.coords).sum()
n_total = cd.coords.size
print(f"\nMissing coordinates: {n_missing}/{n_total} ({n_missing/n_total*100:.1f}%)")

## 2. Visualize Missing Data Pattern

In [ ]:
# Count missing coordinates per trace
import pandas as pd

missing_per_trace = []
for trace_id, group_df in cd.spots.groupby('trace_id'):
    indices = group_df.index
    n_missing_trace = np.isnan(cd.coords[indices]).any(axis=1).sum()
    missing_per_trace.append({
        'trace_id': trace_id,
        'n_missing': n_missing_trace,
        'n_total': len(indices)
    })

df_missing = pd.DataFrame(missing_per_trace)
df_missing['missing_pct'] = df_missing['n_missing'] / df_missing['n_total'] * 100

plt.figure(figsize=(10, 4))
plt.bar(range(len(df_missing)), df_missing['missing_pct'])
plt.axhline(y=df_missing['missing_pct'].mean(), color='r', linestyle='--', 
            label=f'Mean: {df_missing["missing_pct"].mean():.1f}%')
plt.xlabel('Cell')
plt.ylabel('Missing %')
plt.title('Missing Coordinate Rate per Cell')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(df_missing[['trace_id', 'n_missing', 'n_total', 'missing_pct']])

## 3. Method 1: Linear Interpolation

Fast and simple - linearly interpolates along the genomic coordinate.

In [ ]:
from uchrom.im.impute import impute_coordinates

cd_linear = impute_coordinates(cd, method='linear')

n_missing_after = np.isnan(cd_linear.coords).sum()
n_imputed = n_missing - n_missing_after

print(f"Before: {n_missing} missing coordinates")
print(f"After:  {n_missing_after} missing coordinates")
print(f"Imputed: {n_imputed} ({n_imputed/n_missing*100:.1f}% of missing)")

## 4. Method 2: Cubic Spline

Smoother than linear - uses cubic spline interpolation.

In [ ]:
cd_cubic = impute_coordinates(cd, method='cubic')

n_missing_cubic = np.isnan(cd_cubic.coords).sum()
print(f"Cubic imputation: {n_missing - n_missing_cubic} coordinates imputed")

## 5. Method 3: SnapFISH-IMPUTE Algorithm

Most accurate - uses similarity weighting across cells. Requires `sklearn` and `skimage`.

**Note**: This cell will be skipped if dependencies are not installed.

In [ ]:
try:
    cd_snapfish = impute_coordinates(
        cd, 
        method='snapfish',
        max_iter=10,
        verbose=True
    )
    
    n_missing_snapfish = np.isnan(cd_snapfish.coords).sum()
    print(f"\nSnapFISH imputed: {n_missing - n_missing_snapfish} coordinates")
    
except ImportError as e:
    print(f"⚠ SnapFISH requires additional dependencies: {e}")
    print("Install with: pip install scikit-learn scikit-image")
    cd_snapfish = None

## 6. Compare Methods

Visualize one cell before and after imputation.

In [ ]:
# Pick first cell
trace_id = cd.spots['trace_id'].unique()[0]
mask = cd.spots['trace_id'] == trace_id
indices = cd.spots[mask].index

# Get coordinates
coords_orig = cd.coords[indices]
coords_linear = cd_linear.coords[indices]
coords_cubic = cd_cubic.coords[indices]

# Identify missing and imputed points
missing_mask = np.isnan(coords_orig[:, 0])

# Plot
fig = plt.figure(figsize=(15, 4))

# Original
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(coords_orig[~missing_mask, 0], 
           coords_orig[~missing_mask, 1], 
           coords_orig[~missing_mask, 2], 
           c='blue', s=20, label='Original')
ax1.set_title(f'Original\n({missing_mask.sum()} missing)')
ax1.legend()

# Linear
ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(coords_linear[~missing_mask, 0], 
           coords_linear[~missing_mask, 1], 
           coords_linear[~missing_mask, 2], 
           c='blue', s=20, label='Original', alpha=0.5)
ax2.scatter(coords_linear[missing_mask, 0], 
           coords_linear[missing_mask, 1], 
           coords_linear[missing_mask, 2], 
           c='red', s=20, label='Imputed')
ax2.set_title('Linear Interpolation')
ax2.legend()

# Cubic
ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(coords_cubic[~missing_mask, 0], 
           coords_cubic[~missing_mask, 1], 
           coords_cubic[~missing_mask, 2], 
           c='blue', s=20, label='Original', alpha=0.5)
ax3.scatter(coords_cubic[missing_mask, 0], 
           coords_cubic[missing_mask, 1], 
           coords_cubic[missing_mask, 2], 
           c='red', s=20, label='Imputed')
ax3.set_title('Cubic Spline')
ax3.legend()

plt.tight_layout()
plt.show()

## 7. Save Imputed Data

In [ ]:
# Save the imputed data
cd_linear.write('example-data/mESC_Sox2_5cells_imputed_linear.h5cd')
print("✓ Saved: example-data/mESC_Sox2_5cells_imputed_linear.h5cd")

if cd_snapfish is not None:
    cd_snapfish.write('example-data/mESC_Sox2_5cells_imputed_snapfish.h5cd')
    print("✓ Saved: example-data/mESC_Sox2_5cells_imputed_snapfish.h5cd")

## 8. Downstream Analysis

Now you can use the imputed data for downstream analysis like TAD calling, compartment detection, etc.

In [ ]:
# Example: Continue with TAD calling
# from uchrom.strc.tad import get_domains
# ...

print("Imputed ChromData is ready for:")
print("- TAD detection (uchrom.strc.tad)")
print("- Loop calling (uchrom.strc.loop)")
print("- Compartment analysis (uchrom.strc.comp)")
print("- Visualization (uchrom.browser)")

## Summary

### Key Takeaways

1. **Linear/Cubic** - Fast, no dependencies, good for small gaps
2. **SnapFISH** - Most accurate, uses cell similarity, needs sklearn/skimage
3. **All methods** preserve original non-missing coordinates

### When to Use Which Method

- **Linear**: Quick exploratory analysis, <20% missing
- **Cubic**: Smoother than linear, <20% missing
- **SnapFISH**: Best accuracy, >20% missing, have multiple cells

### Performance

| Method | Time (5 cells) | Dependencies |
|--------|----------------|-------------|
| Linear | <1 sec | scipy (core) |
| Cubic | <1 sec | scipy (core) |
| SnapFISH | ~10 sec | sklearn, skimage |

### References

- **SnapFISH-IMPUTE**: Yu et al. (2023) https://github.com/hyuyu104/SnapFISH-IMPUTE
- **Test data**: Huang et al. (2021) *Nature Genetics* doi:10.1038/s41588-021-00863-6